# Example 15 — Radar Range Profile Processing

**Level:** Intermediate

After working through this notebook you will know how to:

- Build a `RadarDataset` with point-scatterer targets at random range bins
- Understand `RadarTarget` ground-truth (range bins, SNRs, waveform label)
- Apply a **matched filter** to maximise SNR for a known pulse shape
- Apply **CA-CFAR** and **OS-CFAR** adaptive threshold detectors
- Visualise and compare detection results against ground truth

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
from plot_helpers import savefig
from spectra.algorithms import ca_cfar, matched_filter, os_cfar
from spectra.datasets.radar import RadarDataset
from spectra.waveforms import LFM, BarkerCodedPulse, PolyphaseCodedPulse

SAMPLE_RATE    = 1e6
NUM_RANGE_BINS = 512
SNR_RANGE      = (5.0, 25.0)
N_SAMPLES      = 200
SEED           = 42
GUARD          = 4
TRAINING       = 16
PFA            = 1e-4

## 1. RadarDataset — waveform pool and ground truth

`RadarDataset` generates synthetic radar range profiles on-the-fly.  Each
item places 0–N point scatterers at random range bins, models them as
scaled, delayed copies of the transmit pulse embedded in AWGN, and returns
a matched-filter range profile as a 1-D tensor.

`RadarTarget` carries per-item ground truth:
- `range_bins` — true target locations (0-indexed)
- `snrs` — per-target SNR in dB
- `waveform_label` — which waveform was used for this item

In [ ]:
waveform_pool = [LFM(), BarkerCodedPulse(), PolyphaseCodedPulse(code_type="p4")]

ds = RadarDataset(
    waveform_pool=waveform_pool,
    num_range_bins=NUM_RANGE_BINS,
    sample_rate=SAMPLE_RATE,
    snr_range=SNR_RANGE,
    num_targets_range=(1, 3),
    num_samples=N_SAMPLES,
    seed=SEED,
)

data, target = ds[0]
print(f"Dataset length: {len(ds)}")
print(f"Sample 0: {target.num_targets} target(s), waveform={target.waveform_label}")
print(f"  Range bins: {target.range_bins}")
print(f"  SNRs (dB):  {target.snrs}")

## 2. Matched Filter

The matched filter maximises output SNR for a known pulse shape **h(t)**
in white Gaussian noise.  It is implemented as convolution with the
time-reversed conjugate of the reference pulse:

**y**[n] = Σ_k **h***[k] · **x**[n+k]

For a point target at delay **d** samples, the peak of |**y**|² appears at
sample **d + L − 1** where **L** is the pulse length.

The output tensor from `RadarDataset` already contains the log-magnitude
range profile normalised to [0, 1], ready for model input.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(data.numpy(), linewidth=0.8, color="steelblue")
for rb in target.range_bins:
    ax.axvline(rb, color="crimson", linestyle="--", linewidth=1.2, label=f"Target @ bin {rb}")
ax.set_xlabel("Range bin")
ax.set_ylabel("Normalised MF power")
ax.set_title(f"Matched-Filter Range Profile — {target.waveform_label}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig("15_range_profile.png")
plt.show()

## 3. CA-CFAR

**Cell-Averaging CFAR** estimates the local noise power from training cells
on either side of the cell under test (CUT), separated by guard cells that
prevent target energy from biasing the estimate.  The adaptive threshold is:

T = α · mean(training cells),  α = N_train · (P_fa^{−1/N_train} − 1)

CA-CFAR achieves the design P_fa for Rayleigh-distributed noise but can
mask closely-spaced targets.

In [ ]:
# Re-generate raw IQ for un-normalised CFAR input
rng = np.random.default_rng((SEED, 0))
wf = LFM()
pulse = wf.generate(num_symbols=4, sample_rate=SAMPLE_RATE, seed=0)[:NUM_RANGE_BINS // 4]
_noise_re = rng.standard_normal(NUM_RANGE_BINS)
_noise_im = rng.standard_normal(NUM_RANGE_BINS)
noise = np.sqrt(0.5) * (_noise_re + 1j * _noise_im)
for rb in target.range_bins:
    snr_lin = 10 ** (target.snrs[0] / 10)
    amp = np.sqrt(snr_lin / (np.mean(np.abs(pulse)**2) + 1e-30))
    if rb + len(pulse) <= NUM_RANGE_BINS:
        noise[rb:rb+len(pulse)] += amp * pulse
mf_raw = np.abs(matched_filter(noise, pulse)[:NUM_RANGE_BINS]) ** 2

det_ca = ca_cfar(mf_raw, guard_cells=GUARD, training_cells=TRAINING, pfa=PFA)
det_ca_bins = np.where(det_ca)[0]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(10 * np.log10(mf_raw + 1e-30), linewidth=0.8)
for rb in target.range_bins:
    axes[0].axvline(rb, color="crimson", linestyle="--", linewidth=1.2)
axes[0].set_ylabel("MF Power (dB)")
axes[0].set_title("Matched Filter Output")
axes[0].grid(True, alpha=0.3)
if len(det_ca_bins) > 0:
    axes[1].stem(
        det_ca_bins, np.ones(len(det_ca_bins)),
        linefmt="C1-", markerfmt="C1o", basefmt="k",
    )
axes[1].set_ylabel("CA-CFAR detections")
axes[1].set_ylim(-0.1, 1.5)
axes[1].set_xlabel("Range bin")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
savefig("15_ca_cfar.png")
plt.show()
print(f"CA-CFAR detections: {det_ca_bins}")
print(f"True target bins:   {target.range_bins}")

## 4. OS-CFAR

**Ordered-Statistics CFAR** uses the k-th ranked training cell as the noise
reference rather than the mean:

T = α · x_(k),  α ≈ k · (P_fa^{−1/(N−k)} − 1)

where x_(k) is the k-th order statistic of the training window.  OS-CFAR
is more robust than CA-CFAR at clutter edges and in the presence of
interfering targets in the training window.

In [ ]:
det_os = os_cfar(mf_raw, guard_cells=GUARD, training_cells=TRAINING, pfa=PFA)
det_os_bins = np.where(det_os)[0]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(10 * np.log10(mf_raw + 1e-30), linewidth=0.8)
for rb in target.range_bins:
    axes[0].axvline(rb, color="crimson", linestyle="--", linewidth=1.2)
axes[0].set_ylabel("MF Power (dB)")
axes[0].set_title("CA-CFAR vs OS-CFAR")
axes[0].grid(True, alpha=0.3)
if len(det_ca_bins) > 0:
    axes[1].stem(
        det_ca_bins, np.ones(len(det_ca_bins)),
        linefmt="C1-", markerfmt="C1o", basefmt="k",
    )
axes[1].set_ylabel("CA-CFAR")
axes[1].set_ylim(-0.1, 1.5)
axes[1].grid(True, alpha=0.3)
if len(det_os_bins) > 0:
    axes[2].stem(
        det_os_bins, np.ones(len(det_os_bins)),
        linefmt="C2-", markerfmt="C2o", basefmt="k",
    )
axes[2].set_ylabel("OS-CFAR")
axes[2].set_ylim(-0.1, 1.5)
axes[2].set_xlabel("Range bin")
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
savefig("15_cfar_detections.png")
plt.show()
print(f"OS-CFAR detections: {det_os_bins}")
print(f"True target bins:   {target.range_bins}")

## 5. Waveform Mix

The dataset draws waveforms uniformly from the pool.  The bar chart below
shows the waveform distribution over the first 100 samples.

In [ ]:
from collections import Counter

labels_seen = [ds[i][1].waveform_label for i in range(min(100, len(ds)))]
counts = Counter(labels_seen)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(counts.keys(), counts.values(), color="steelblue")
ax.set_ylabel("Count")
ax.set_title("Waveform distribution in first 100 samples")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
savefig("15_waveform_distribution.png")
plt.show()